# Classification Algorithms for Beginners

## What You'll Learn

Classification is used to predict **categories or classes**. In this notebook, you'll master:

1. **Logistic Regression** - Linear classification
2. **K-Nearest Neighbors (KNN)** - Instance-based learning
3. **Decision Tree Classifier** - Rule-based classification
4. **Random Forest Classifier** - Ensemble method
5. **Support Vector Machine (SVM)** - Maximum margin classifier
6. **Naive Bayes** - Probabilistic classifier

**Real-World Applications:**
- Email spam detection
- Disease diagnosis
- Customer churn prediction
- Sentiment analysis
- Image classification

---

## Setup: Import Libraries

In [ ]:
# Install required packages (uncomment if needed)
# !pip install numpy pandas matplotlib seaborn scikit-learn

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, roc_auc_score)

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries imported successfully!")

## Understanding Classification

### Binary vs Multi-class Classification

| Type | Classes | Example |
|------|---------|----------|
| **Binary** | 2 | Spam/Not Spam, Yes/No, Positive/Negative |
| **Multi-class** | 3+ | Dog/Cat/Bird, Low/Medium/High |

## Load Dataset: Breast Cancer Wisconsin

**Goal:** Predict whether a tumor is **malignant** (cancerous) or **benign** (non-cancerous).

This is a **binary classification** problem with medical importance!

In [ ]:
from sklearn.datasets import load_breast_cancer

# Load dataset
cancer = load_breast_cancer()

# Create DataFrame
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print(f"Dataset Shape: {df.shape}")
print(f"\nFeatures: {len(cancer.feature_names)} measurements")
print(f"\nTarget Classes: {cancer.target_names}")
print(f"  - 0: Malignant (Cancerous)")
print(f"  - 1: Benign (Non-cancerous)")
df.head()

## Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
print("Class Distribution:")
print(df['diagnosis'].value_counts())

plt.figure(figsize=(8, 5))
colors = ['#FF6B6B', '#4ECDC4']
df['diagnosis'].value_counts().plot(kind='bar', color=colors, edgecolor='black')
plt.title('Distribution of Tumor Types')
plt.xlabel('Diagnosis')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Select key features for visualization
key_features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area',
                'mean smoothness', 'mean compactness']

# Compare feature distributions by diagnosis
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for idx, feature in enumerate(key_features):
    ax = axes[idx // 3, idx % 3]
    
    for diagnosis, color in zip(['Malignant', 'Benign'], colors):
        data = df[df['diagnosis'] == diagnosis][feature]
        ax.hist(data, bins=20, alpha=0.6, label=diagnosis, color=color)
    
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')
    ax.set_title(f'Distribution of {feature}')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target
feature_cols = [col for col in df.columns if col not in ['target', 'diagnosis']]
correlations = df[feature_cols].corrwith(df['target']).sort_values()

plt.figure(figsize=(12, 10))
colors = ['red' if x < 0 else 'green' for x in correlations]
plt.barh(range(len(correlations)), correlations.values, color=colors)
plt.yticks(range(len(correlations)), correlations.index)
plt.xlabel('Correlation with Benign (positive = more likely benign)')
plt.title('Feature Correlation with Diagnosis')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot for top features
top_features = ['mean radius', 'mean concave points', 'worst perimeter', 'diagnosis']
sns.pairplot(df[top_features], hue='diagnosis', palette={'Malignant': '#FF6B6B', 'Benign': '#4ECDC4'})
plt.suptitle('Pairplot of Top Features', y=1.02)
plt.show()

## Data Preparation

In [ ]:
# Prepare features and target
X = df.drop(['target', 'diagnosis'], axis=1)
y = df['target']  # 0 = Malignant, 1 = Benign

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nClass distribution in training set:")
print(y_train.value_counts())

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully!")

## Classification Evaluation Metrics

Understanding metrics is crucial for classification:

| Metric | Formula | When to Use |
|--------|---------|-------------|
| **Accuracy** | (TP+TN)/Total | Balanced classes |
| **Precision** | TP/(TP+FP) | Cost of false positives is high |
| **Recall** | TP/(TP+FN) | Cost of false negatives is high |
| **F1 Score** | 2*(P*R)/(P+R) | Balance precision and recall |
| **AUC-ROC** | Area under ROC curve | Overall model quality |

**For medical diagnosis:** Recall is often more important (don't miss cancer cases!)

In [ ]:
# Helper function to evaluate classification models
def evaluate_classifier(model, X_train, X_test, y_train, y_test, model_name):
    """
    Train classifier and return evaluation metrics
    """
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Get probabilities if available
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    else:
        y_prob = None
        auc = None
    
    # Calculate metrics
    results = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'AUC': auc
    }
    
    return results, y_pred, y_prob

---

# Algorithm 1: Logistic Regression

## Linear Classification

**Concept:** Despite its name, Logistic Regression is for classification, not regression!

**How it works:**
1. Compute linear combination: z = b₀ + b₁x₁ + b₂x₂ + ...
2. Apply sigmoid function: P(y=1) = 1 / (1 + e^(-z))
3. Output probability between 0 and 1
4. Classify based on threshold (usually 0.5)

**The Sigmoid Function:**
- Maps any input to [0, 1]
- Perfect for probability output

In [ ]:
# Visualize sigmoid function
plt.figure(figsize=(10, 6))
z = np.linspace(-10, 10, 100)
sigmoid = 1 / (1 + np.exp(-z))

plt.plot(z, sigmoid, linewidth=3, color='blue')
plt.axhline(y=0.5, color='red', linestyle='--', label='Decision boundary (0.5)')
plt.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('z (linear combination)')
plt.ylabel('Probability P(y=1)')
plt.title('Sigmoid Function: Converting Values to Probabilities')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_results, lr_pred, lr_prob = evaluate_classifier(
    lr_model, X_train_scaled, X_test_scaled, y_train, y_test, 'Logistic Regression'
)

print("=" * 50)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 50)
for key, value in lr_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Logistic Regression: Confusion Matrix')
plt.show()

print("\nInterpretation:")
print(f"True Negatives (correctly predicted Malignant): {cm[0,0]}")
print(f"False Positives (Malignant predicted as Benign): {cm[0,1]}")
print(f"False Negatives (Benign predicted as Malignant): {cm[1,0]}")
print(f"True Positives (correctly predicted Benign): {cm[1,1]}")

---

# Algorithm 2: K-Nearest Neighbors (KNN)

## Instance-Based Learning

**Concept:** Classify based on the majority class of k nearest neighbors.

**How it works:**
1. Calculate distance to all training points
2. Find k nearest neighbors
3. Vote: majority class wins

**Key Parameter:** k (number of neighbors)
- Small k: More complex, prone to noise
- Large k: Smoother, may miss local patterns

In [ ]:
# Visualize KNN concept with 2D example
from sklearn.datasets import make_classification

# Generate simple 2D data
X_demo, y_demo = make_classification(n_samples=100, n_features=2, n_redundant=0,
                                      n_informative=2, random_state=42, n_clusters_per_class=1)

plt.figure(figsize=(10, 6))
plt.scatter(X_demo[y_demo==0, 0], X_demo[y_demo==0, 1], c='red', label='Class 0', s=50)
plt.scatter(X_demo[y_demo==1, 0], X_demo[y_demo==1, 1], c='blue', label='Class 1', s=50)

# New point to classify
new_point = [0, 0]
plt.scatter(*new_point, c='green', s=200, marker='*', label='New Point')

# Draw circle for k=3
circle = plt.Circle(new_point, 1.5, fill=False, linestyle='--', color='green', linewidth=2)
plt.gca().add_patch(circle)

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('KNN: Classify Based on Nearest Neighbors')
plt.legend()
plt.axis('equal')
plt.show()

In [ ]:
# Find optimal k
k_values = range(1, 31)
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

# Plot
plt.figure(figsize=(10, 6))
plt.plot(k_values, cv_scores, 'bo-', linewidth=2, markersize=6)
optimal_k = k_values[np.argmax(cv_scores)]
plt.axvline(x=optimal_k, color='red', linestyle='--', label=f'Optimal k={optimal_k}')
plt.xlabel('k (Number of Neighbors)')
plt.ylabel('Cross-Validation Accuracy')
plt.title('KNN: Finding Optimal k')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Optimal k: {optimal_k}")

In [ ]:
# KNN with optimal k
knn_model = KNeighborsClassifier(n_neighbors=optimal_k)
knn_results, knn_pred, knn_prob = evaluate_classifier(
    knn_model, X_train_scaled, X_test_scaled, y_train, y_test, f'KNN (k={optimal_k})'
)

print("=" * 50)
print(f"KNN RESULTS (k={optimal_k})")
print("=" * 50)
for key, value in knn_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

---

# Algorithm 3: Decision Tree Classifier

## Rule-Based Learning

**Concept:** Learn if-then-else rules from data.

**How it works:**
1. Find best feature/threshold to split data
2. Split into child nodes
3. Repeat until stopping criteria
4. Classify by majority in leaf node

**Advantages:**
- Easy to interpret
- No scaling needed
- Handles both numerical and categorical features

In [ ]:
# Decision Tree with different depths
depths = [2, 4, 6, 10, None]
dt_results_list = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    y_pred = dt.predict(X_test)
    
    dt_results_list.append({
        'Max Depth': depth if depth else 'None',
        'Train Accuracy': dt.score(X_train, y_train),
        'Test Accuracy': accuracy_score(y_test, y_pred)
    })

dt_comparison = pd.DataFrame(dt_results_list)
print("Decision Tree Results with Different Depths:")
print(dt_comparison)

In [ ]:
# Best Decision Tree
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_results, dt_pred, dt_prob = evaluate_classifier(
    dt_model, X_train, X_test, y_train, y_test, 'Decision Tree'
)

print("=" * 50)
print("DECISION TREE RESULTS (max_depth=4)")
print("=" * 50)
for key, value in dt_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Visualize the decision tree
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))
plot_tree(dt_model, 
          feature_names=X.columns,
          class_names=['Malignant', 'Benign'],
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('Decision Tree Visualization')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
dt_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(dt_importance['Feature'], dt_importance['Importance'], color='teal')
plt.xlabel('Feature Importance')
plt.title('Decision Tree: Top 10 Features')
plt.gca().invert_yaxis()
plt.show()

---

# Algorithm 4: Random Forest Classifier

## Ensemble of Decision Trees

**Concept:** Many trees vote on the final classification.

**How it works:**
1. Create multiple decision trees (bootstrap sampling)
2. Each tree uses random feature subset
3. Final prediction = majority vote

**Advantages:**
- Reduces overfitting
- More robust than single tree
- Handles high-dimensional data well

In [ ]:
# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,     # Number of trees
    max_depth=10,         # Maximum depth
    min_samples_split=5,  # Minimum samples to split
    random_state=42,
    n_jobs=-1             # Use all CPU cores
)

rf_results, rf_pred, rf_prob = evaluate_classifier(
    rf_model, X_train, X_test, y_train, y_test, 'Random Forest'
)

print("=" * 50)
print("RANDOM FOREST RESULTS")
print("=" * 50)
for key, value in rf_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Random Forest feature importance
rf_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(rf_importance['Feature'], rf_importance['Importance'], color='forestgreen')
plt.xlabel('Feature Importance')
plt.title('Random Forest: Top 10 Features')
plt.gca().invert_yaxis()
plt.show()

---

# Algorithm 5: Support Vector Machine (SVM)

## Maximum Margin Classifier

**Concept:** Find the hyperplane that maximizes the margin between classes.

**How it works:**
1. Find the decision boundary (hyperplane)
2. Maximize distance to nearest points (support vectors)
3. Use kernel trick for non-linear separation

**Kernels:**
- Linear: Straight line separation
- RBF (Gaussian): Non-linear, flexible
- Polynomial: Polynomial decision boundary

In [ ]:
# Visualize SVM concept
from sklearn.datasets import make_blobs

# Generate 2D data
X_svm, y_svm = make_blobs(n_samples=100, centers=2, random_state=42, cluster_std=1.5)

# Fit SVM
svm_demo = SVC(kernel='linear')
svm_demo.fit(X_svm, y_svm)

# Plot
plt.figure(figsize=(10, 6))

# Plot data
plt.scatter(X_svm[y_svm==0, 0], X_svm[y_svm==0, 1], c='red', s=50, label='Class 0')
plt.scatter(X_svm[y_svm==1, 0], X_svm[y_svm==1, 1], c='blue', s=50, label='Class 1')

# Plot support vectors
plt.scatter(svm_demo.support_vectors_[:, 0], svm_demo.support_vectors_[:, 1],
            s=200, facecolors='none', edgecolors='black', linewidth=2, label='Support Vectors')

# Plot decision boundary
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
Z = svm_demo.decision_function(xy).reshape(XX.shape)

ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1], alpha=0.5,
           linestyles=['--', '-', '--'])

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('SVM: Maximum Margin Classification')
plt.legend()
plt.show()

In [ ]:
# Compare SVM kernels
kernels = ['linear', 'rbf', 'poly']
svm_results_list = []

for kernel in kernels:
    svm = SVC(kernel=kernel, random_state=42, probability=True)
    svm.fit(X_train_scaled, y_train)
    y_pred = svm.predict(X_test_scaled)
    
    svm_results_list.append({
        'Kernel': kernel,
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)
    })

svm_comparison = pd.DataFrame(svm_results_list)
print("SVM Results with Different Kernels:")
print(svm_comparison)

In [ ]:
# Best SVM model (RBF kernel)
svm_model = SVC(kernel='rbf', random_state=42, probability=True)
svm_results, svm_pred, svm_prob = evaluate_classifier(
    svm_model, X_train_scaled, X_test_scaled, y_train, y_test, 'SVM (RBF)'
)

print("=" * 50)
print("SVM RESULTS (RBF Kernel)")
print("=" * 50)
for key, value in svm_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

---

# Algorithm 6: Naive Bayes

## Probabilistic Classification

**Concept:** Apply Bayes' theorem with "naive" independence assumption.

**Bayes' Theorem:**
P(Class|Features) = P(Features|Class) × P(Class) / P(Features)

**"Naive" Assumption:**
Features are independent given the class (often not true, but works well in practice!)

**Advantages:**
- Very fast
- Works well with limited data
- Great for text classification

In [ ]:
# Naive Bayes
nb_model = GaussianNB()
nb_results, nb_pred, nb_prob = evaluate_classifier(
    nb_model, X_train_scaled, X_test_scaled, y_train, y_test, 'Naive Bayes'
)

print("=" * 50)
print("NAIVE BAYES RESULTS")
print("=" * 50)
for key, value in nb_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

---

# Model Comparison Summary

In [ ]:
# Compile all results
all_results = pd.DataFrame([
    lr_results,
    knn_results,
    dt_results,
    rf_results,
    svm_results,
    nb_results
])

print("=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)
print(all_results.round(4).to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy and F1 comparison
models = all_results['Model']
x_pos = np.arange(len(models))

axes[0].bar(x_pos - 0.2, all_results['Accuracy'], 0.4, label='Accuracy', color='steelblue')
axes[0].bar(x_pos + 0.2, all_results['F1 Score'], 0.4, label='F1 Score', color='coral')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(models, rotation=45, ha='right')
axes[0].set_ylabel('Score')
axes[0].set_title('Accuracy vs F1 Score Comparison')
axes[0].legend()
axes[0].set_ylim(0.8, 1.0)

# AUC comparison
auc_values = all_results['AUC'].fillna(0)
axes[1].bar(models, auc_values, color='teal')
axes[1].set_xticklabels(models, rotation=45, ha='right')
axes[1].set_ylabel('AUC Score')
axes[1].set_title('AUC-ROC Comparison')
axes[1].set_ylim(0.9, 1.0)

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(10, 8))

# Plot ROC curves for models with probabilities
models_probs = [
    ('Logistic Regression', lr_prob),
    ('KNN', knn_prob),
    ('Decision Tree', dt_prob),
    ('Random Forest', rf_prob),
    ('SVM', svm_prob),
    ('Naive Bayes', nb_prob)
]

for name, prob in models_probs:
    if prob is not None:
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc = roc_auc_score(y_test, prob)
        plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

---

## Key Takeaways

| Algorithm | Best For | Pros | Cons |
|-----------|----------|------|------|
| **Logistic Regression** | Linear separability | Fast, interpretable, probabilities | Linear only |
| **KNN** | Simple patterns, small data | No training, flexible | Slow prediction, curse of dimensionality |
| **Decision Tree** | Interpretability needed | Easy to explain, no scaling | Overfits easily |
| **Random Forest** | General purpose | Robust, accurate | Slower, black box |
| **SVM** | High-dimensional data | Effective in high dimensions | Slow on large datasets |
| **Naive Bayes** | Text, quick baseline | Very fast, works with limited data | Strong independence assumption |

---

## Practice Exercises

1. Try multi-class classification with the Iris dataset
2. Experiment with hyperparameter tuning using GridSearchCV
3. Handle imbalanced classes using techniques like SMOTE
4. Compare model performance on a different dataset from Kaggle

In [ ]:
# Exercise: Multi-class Classification with Iris
from sklearn.datasets import load_iris

# Your code here:
# iris = load_iris()
# ...